In [58]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# -----------------------
# Load Data
# -----------------------

df = pd.read_csv("/kaggle/input/datasets/ankushpanday1/air-quality-data-in-india-2015-2024/city_hour.csv")

df = df[df["City"].str.lower().str.strip()=="mumbai"].copy()

df = df.sort_values("Datetime")
df["Datetime"] = pd.to_datetime(df["Datetime"])
df["Year"] = df["Datetime"].dt.year
df["Month"] = df["Datetime"].dt.month
df["Day"] = df["Datetime"].dt.day
df["Hour"] = df["Datetime"].dt.hour
df["DayOfWeek"] = df["Datetime"].dt.dayofweek
df["WeekOfYear"] = df["Datetime"].dt.isocalendar().week.astype(int)
df["Quarter"] = df["Datetime"].dt.quarter

df = df[df["PM2.5"].notna()]

# -----------------------
# Features
# -----------------------

features = [
    "City",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene",
    "Xylene",
    "Year",
    "Month",
    "Day",
    "Hour",
    "DayOfWeek",
    "WeekOfYear",
    "Quarter"
]

target = "PM2.5"

X = df[features]
y = df[target]
# -----------------------
# Preprocessing
# -----------------------

numeric_features = [
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene",
    "Xylene",
    "Year",
    "Month",
    "Day",
    "Hour",
    "DayOfWeek",
    "WeekOfYear",
    "Quarter"
]

categorical_features = ["City"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [59]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
from catboost import CatBoostRegressor

models = {

    "Linear Regression":
        LinearRegression(),

    "Decision Tree":
        DecisionTreeRegressor(
            random_state=42
        ),

    "Random Forest":
        RandomForestRegressor(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        ),

    "XGBoost":
        XGBRegressor(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            tree_method="hist",      # Faster on CPU
            n_jobs=-1,
            random_state=42
        ),

    "CatBoost":
        CatBoostRegressor(
            iterations=200,
            depth=6,
            learning_rate=0.05,
            verbose=50,              # Shows progress
            random_seed=42
        )
}

In [60]:
df = df.sort_values("Datetime")

split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

In [61]:
print(df.shape)

(87649, 23)


In [62]:
import time

results = []

for name, model in models.items():

    print(f"\nTraining {name}...")
    start = time.time()

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_test)

    end = time.time()

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    print(f"✓ {name} completed in {end-start:.1f} seconds")

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Time (s)": round(end-start, 2)
    })

results = pd.DataFrame(results)
print(results.sort_values("R2", ascending=False))


Training Linear Regression...
✓ Linear Regression completed in 0.3 seconds

Training Decision Tree...
✓ Decision Tree completed in 3.5 seconds

Training Random Forest...
✓ Random Forest completed in 85.5 seconds

Training XGBoost...
✓ XGBoost completed in 1.5 seconds

Training CatBoost...
0:	learn: 144.4324100	total: 13ms	remaining: 2.59s
50:	learn: 144.0420500	total: 476ms	remaining: 1.39s
100:	learn: 143.6687265	total: 944ms	remaining: 925ms
150:	learn: 143.3322219	total: 1.41s	remaining: 458ms
199:	learn: 143.0069226	total: 1.85s	remaining: 0us
✓ CatBoost completed in 2.2 seconds
               Model         MAE        RMSE        R2  Time (s)
0  Linear Regression  124.177816  143.465522 -0.000289      0.31
4           CatBoost  124.237003  143.545117 -0.001399      2.21
3            XGBoost  124.519785  144.001299 -0.007774      1.47
2      Random Forest  124.696594  144.444237 -0.013983     85.54
1      Decision Tree  167.208922  204.766760 -1.037742      3.51


In [63]:
print(features)

['City', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'Year', 'Month', 'Day', 'Hour', 'DayOfWeek', 'WeekOfYear', 'Quarter']
